<a href="https://colab.research.google.com/github/Obulreddy0/pyspark-practice/blob/master/Day2_practise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pyspark -q

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Day1_Practice") \
    .master("local[*]") \
    .getOrCreate()

print(f"✅ Spark {spark.version} ready!")

✅ Spark 4.0.4 ready!


In [2]:
!git clone https://github.com/Obulreddy0/pyspark-practice.git

customers_df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("pyspark-practice/data/customers.csv")

customers_df.show()

Cloning into 'pyspark-practice'...
remote: Enumerating objects: 39, done.
remote: Counting objects: 100% (39/39), done.
remote: Compressing objects: 100% (34/34), done.
remote: Total 39 (delta 12), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (39/39), 14.78 KiB | 841.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.
+-----------+-------------+-----------------+---------+-----------+----+-----------+---------+
|customer_id|         name|            email|     city|      state| age|signup_date|is_active|
+-----------+-------------+-----------------+---------+-----------+----+-----------+---------+
|       C001| Rahul Sharma|  rahul@email.com|   Mumbai|Maharashtra|  28| 2023-01-15|     true|
|       C002|  Priya Patel|  priya@email.com|Ahmedabad|    Gujarat|  34| 2023-02-20|     true|
|       C003|   Amit Kumar|   amit@email.com|    Delhi|      Delhi|  45| 2022-11-10|    false|
|       C004|  Sneha Reddy|  sneha@email.com|Hyderabad|  Telangana|  29| 2023-03-05| 

In [3]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("pyspark-practice/data/orders.csv")

df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- status: string (nullable = true)



In [4]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("status", StringType(), True),
])

df = spark.read \
    .option("header", "true") \
    .schema(orders_schema) \
    .csv("pyspark-practice/data/orders.csv")

df.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- status: string (nullable = true)



In [8]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

spark = SparkSession.builder.appName("ETL_Example").master("local[*]").getOrCreate()

# 1. Define schema
schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("status", StringType(), True),
])

# 2. Read CSV with explicit schema
orders_df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("pyspark-practice/data/orders.csv")

# 3. Quick check
print(f"Rows: {orders_df.count()}, Columns: {len(orders_df.columns)}")
orders_df.show(5)

# 4. Write as Parquet (production format)
orders_df.write.mode("overwrite").parquet("output/orders_parquet")

print("✅ CSV → Parquet conversion done!")


Rows: 25, Columns: 8
+--------+-----------+----------+----------+--------+----------+--------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount|   status|
+--------+-----------+----------+----------+--------+----------+--------+---------+
|   O1001|       C001|      P101|2023-06-01|       2|    1500.0|     0.1|delivered|
|   O1002|       C002|      P102|2023-06-02|       1|    2500.0|     0.0|delivered|
|   O1003|       C001|      P103|2023-06-03|       3|     800.0|    0.05|  shipped|
|   O1004|       C003|      P101|2023-06-05|       1|    1500.0|     0.0|cancelled|
|   O1005|       C004|      P104|2023-06-07|       2|    3200.0|    0.15|delivered|
+--------+-----------+----------+----------+--------+----------+--------+---------+
only showing top 5 rows
✅ CSV → Parquet conversion done!


In [10]:
# Read products.csv using inferSchema. Print the schema. Now read it again with a manual schema using StructType. Compare both schemas — are the types the same?

df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("pyspark-practice/data/products.csv")

df.printSchema()

schema = StructType([
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("brand", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("stock_qty", IntegerType(), True),
])

df = spark.read \
    .option("header", "true") \
    .schema(schema) \
    .csv("pyspark-practice/data/products.csv")

df.printSchema()


root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_qty: integer (nullable = true)

root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- category: string (nullable = true)
 |-- brand: string (nullable = true)
 |-- price: double (nullable = true)
 |-- stock_qty: integer (nullable = true)



In [11]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, BooleanType

schema = StructType([
    StructField("txn_id", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("txn_type", StringType(), True),
    StructField("txn_date", StringType(), True),
    StructField("is_fraud", BooleanType(), True),
])

data = [
    ("T001", 15000.50, "debit", "2023-06-01", False),
    ("T002", 250000.00, "credit", "2023-06-02", False),
    ("T003", 999999.99, "debit", "2023-06-03", True),
    ("T004", 500.00, "debit", "2023-06-04", False),
    ("T005", 75000.00, "credit", "2023-06-05", True),
]

df = spark.createDataFrame(data, schema)
df.show()
df.printSchema()

+------+---------+--------+----------+--------+
|txn_id|   amount|txn_type|  txn_date|is_fraud|
+------+---------+--------+----------+--------+
|  T001|  15000.5|   debit|2023-06-01|   false|
|  T002| 250000.0|  credit|2023-06-02|   false|
|  T003|999999.99|   debit|2023-06-03|    true|
|  T004|    500.0|   debit|2023-06-04|   false|
|  T005|  75000.0|  credit|2023-06-05|    true|
+------+---------+--------+----------+--------+

root
 |-- txn_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- txn_type: string (nullable = true)
 |-- txn_date: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)



In [12]:
# Read customers
customers_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("pyspark-practice/data/customers.csv")

# Write as Parquet
customers_df.write.mode("overwrite").parquet("output/customers_parquet")
print("✅ Written as Parquet")

# Write as JSON
customers_df.write.mode("overwrite").json("output/customers_json")
print("✅ Written as JSON")

# Read back Parquet and verify schema
parquet_df = spark.read.parquet("output/customers_parquet")

print("\n=== Original Schema ===")
customers_df.printSchema()

print("=== Parquet Schema ===")
parquet_df.printSchema()

✅ Written as Parquet
✅ Written as JSON

=== Original Schema ===
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)

=== Parquet Schema ===
root
 |-- customer_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- is_active: boolean (nullable = true)



In [13]:
# Read orders.csv with this intentionally wrong schema and observe what happens:

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

wrong_schema = StructType([
    StructField("order_id", IntegerType(), True),     # WRONG! "O1001" is not an integer
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("quantity", StringType(), True),       # Works but wrong type
    StructField("unit_price", DoubleType(), True),
    StructField("discount", DoubleType(), True),
    StructField("status", StringType(), True),
])

df = spark.read \
    .option("header", "true") \
    .schema(wrong_schema) \
    .csv("pyspark-practice/data/orders.csv")

df.show(5)

+--------+-----------+----------+----------+--------+----------+--------+---------+
|order_id|customer_id|product_id|order_date|quantity|unit_price|discount|   status|
+--------+-----------+----------+----------+--------+----------+--------+---------+
|    NULL|       C001|      P101|2023-06-01|       2|    1500.0|     0.1|delivered|
|    NULL|       C002|      P102|2023-06-02|       1|    2500.0|     0.0|delivered|
|    NULL|       C001|      P103|2023-06-03|       3|     800.0|    0.05|  shipped|
|    NULL|       C003|      P101|2023-06-05|       1|    1500.0|     0.0|cancelled|
|    NULL|       C004|      P104|2023-06-07|       2|    3200.0|    0.15|delivered|
+--------+-----------+----------+----------+--------+----------+--------+---------+
only showing top 5 rows


Explanation:

1.   order_id becomes null — "O1001" can't be converted to Integer, so Spark puts null
2.   Spark does NOT throw an error — it silently fails with nulls

3.   quantity works as StringType but you lose the ability to do math on it

🎯 Lesson:

  Wrong schemas don't crash — they silently corrupt your data. This is worse than an error! Always validate your schema after reading.



In [14]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Define schema
txn_schema = StructType([
    StructField("txn_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("txn_date", StringType(), True),
    StructField("txn_type", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("channel", StringType(), True),
    StructField("status", StringType(), True),
])

# 2. Read CSV
txn_df = spark.read \
    .option("header", "true") \
    .schema(txn_schema) \
    .csv("pyspark-practice/data/transactions.csv")

csv_count = txn_df.count()
print(f"CSV row count: {csv_count}")

# 3. Write as Parquet
txn_df.write.mode("overwrite").parquet("output/txn_parquet")
print("✅ Written as Parquet")

# 4. Read Parquet back
txn_parquet_df = spark.read.parquet("output/txn_parquet")

parquet_count = txn_parquet_df.count()
print(f"Parquet row count: {parquet_count}")

# 5. Verify
if csv_count == parquet_count:
    print(f"✅ Match! Both have {csv_count} rows")
else:
    print(f"❌ Mismatch! CSV={csv_count}, Parquet={parquet_count}")

CSV row count: 20
✅ Written as Parquet
Parquet row count: 20
✅ Match! Both have 20 rows


In [15]:
orders_df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("pyspark-practice/data/orders.csv")
customers_df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("pyspark-practice/data/customers.csv")
txn_df = spark.read.option("header", "true").option("inferSchema", "true") \
    .csv("pyspark-practice/data/transactions.csv")

In [16]:
# DESCRIBE TABLE orders;

# PySpark equivalent
orders_df.printSchema()

# Or as a DataFrame (closer to SQL DESCRIBE)
orders_df.dtypes   # Returns list of (column_name, type) tuples

# SELECT * FROM transactions LIMIT 10;
txn_df.show(10)

# SELECT COUNT(*) FROM information_schema.columns WHERE table_name = 'customers';
print(f"Number of columns: {len(customers_df.columns)}")   # → 8

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- status: string (nullable = true)

+------+-----------+----------+--------+--------+-------+-------+
|txn_id|customer_id|  txn_date|txn_type|  amount|channel| status|
+------+-----------+----------+--------+--------+-------+-------+
| T5001|       C001|2023-06-01|   debit|  1200.5| online|success|
| T5002|       C002|2023-06-01|  credit| 50000.0| branch|success|
| T5003|       C003|2023-06-02|   debit|  3400.0|    atm|success|
| T5004|       C001|2023-06-03|   debit|   750.0| online| failed|
| T5005|       C004|2023-06-04|  credit| 25000.0| online|success|
| T5006|       C005|2023-06-05|   debit|  8900.0|    pos|success|
| T5007|       C002|2023-06-06|   debit|  1500.0|    atm|success|
| T5008|

Exercise 7: Dynamic Schema Builder 🧠

Write a function that takes any CSV file path and returns:

A StructType schema with all columns as StringType (safest approach)

The DataFrame loaded with that schema

In [17]:
from pyspark.sql.types import StructType, StructField, StringType

def safe_read_csv(spark, file_path):
    """
    Reads any CSV safely by treating ALL columns as StringType.
    Step 1: Read just header to get column names
    Step 2: Build StructType with all StringType
    Step 3: Load full file with that schema
    """

    # Step 1: Read first row to get column names
    temp_df = spark.read \
        .option("header", "true") \
        .csv(file_path)

    column_names = temp_df.columns

    # Step 2: Build safe schema — all StringType
    safe_schema = StructType([
        StructField(col_name, StringType(), True)
        for col_name in column_names
    ])

    # Step 3: Read with safe schema
    df = spark.read \
        .option("header", "true") \
        .schema(safe_schema) \
        .csv(file_path)

    return safe_schema, df


# Test with all 4 files
files = [
    "pyspark-practice/data/customers.csv",
    "pyspark-practice/data/orders.csv",
    "pyspark-practice/data/transactions.csv",
    "pyspark-practice/data/products.csv",
]

for file_path in files:
    schema, df = safe_read_csv(spark, file_path)
    file_name = file_path.split("/")[-1]
    print(f"\n=== {file_name} ===")
    print(f"Columns: {df.columns}")
    print(f"Rows: {df.count()}")
    df.show(3)


=== customers.csv ===
Columns: ['customer_id', 'name', 'email', 'city', 'state', 'age', 'signup_date', 'is_active']
Rows: 15
+-----------+------------+---------------+---------+-----------+---+-----------+---------+
|customer_id|        name|          email|     city|      state|age|signup_date|is_active|
+-----------+------------+---------------+---------+-----------+---+-----------+---------+
|       C001|Rahul Sharma|rahul@email.com|   Mumbai|Maharashtra| 28| 2023-01-15|     true|
|       C002| Priya Patel|priya@email.com|Ahmedabad|    Gujarat| 34| 2023-02-20|     true|
|       C003|  Amit Kumar| amit@email.com|    Delhi|      Delhi| 45| 2022-11-10|    false|
+-----------+------------+---------------+---------+-----------+---+-----------+---------+
only showing top 3 rows

=== orders.csv ===
Columns: ['order_id', 'customer_id', 'product_id', 'order_date', 'quantity', 'unit_price', 'discount', 'status']
Rows: 25
+--------+-----------+----------+----------+--------+----------+-------

In [18]:
# Later you cast explicitly:
from pyspark.sql.functions import col
df = df.withColumn("price", col("price").cast("double"))

Exercise 8: Format Comparison Benchmark ⏱️

* Read orders.csv into a DataFrame
* Write it in 3 formats: CSV, JSON, Parquet
* Check the file sizes on disk for each format



In [19]:
import time

# Read source data
orders_df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("pyspark-practice/data/orders.csv")

# ========== WRITE in 3 formats ==========
orders_df.write.mode("overwrite").option("header", "true").csv("output/orders_csv")
orders_df.write.mode("overwrite").json("output/orders_json")
orders_df.write.mode("overwrite").parquet("output/orders_parquet")
print("✅ Written in all 3 formats")

# ========== CHECK FILE SIZES ==========
import subprocess
for fmt in ["orders_csv", "orders_json", "orders_parquet"]:
    result = subprocess.run(["du", "-sh", f"output/{fmt}"], capture_output=True, text=True)
    print(result.stdout.strip())

# In Colab you can also do:
# !du -sh output/orders_csv
# !du -sh output/orders_json
# !du -sh output/orders_parquet

# ========== BENCHMARK READ TIMES ==========
results = []

for fmt, reader in [
    ("CSV", lambda: spark.read.option("header","true").option("inferSchema","true").csv("output/orders_csv")),
    ("JSON", lambda: spark.read.json("output/orders_json")),
    ("Parquet", lambda: spark.read.parquet("output/orders_parquet")),
]:
    start = time.time()
    df = reader()
    df.count()    # Force execution (remember: lazy evaluation!)
    elapsed = time.time() - start
    results.append((fmt, elapsed))
    print(f"{fmt:10s} → Read time: {elapsed:.4f} sec")

# ========== SUMMARY TABLE ==========
summary_data = [(fmt, round(t, 4)) for fmt, t in results]
summary_df = spark.createDataFrame(summary_data, ["Format", "ReadTimeSec"])
summary_df.show()

✅ Written in all 3 formats
16K	output/orders_csv
16K	output/orders_json
16K	output/orders_parquet
CSV        → Read time: 0.5320 sec
JSON       → Read time: 0.4888 sec
Parquet    → Read time: 0.3889 sec
+-------+-----------+
| Format|ReadTimeSec|
+-------+-----------+
|    CSV|      0.532|
|   JSON|     0.4888|
|Parquet|     0.3889|
+-------+-----------+

